# 10 – Payoff Matrix Construction

This notebook converts the measured attack × defense results into attacker and defender payoff matrices. It supports `quick`, `pilot`, and `full` metric files and automatically selects the most complete available run.

For attack strategy $A_i$ and defense $D_j$, the default defender payoff is

$$U_D(A_i,D_j)=w_s B_{ij}-w_f FPR_j-w_t T_{ij}-w_l L_j,$$

where $B_{ij}$ is the measured attack block rate, and the token and latency terms are min–max normalized costs. The aggregate defender F1 score is retained as a diagnostic metric but is not inserted into every game cell because it already combines detection and false-positive behavior and is not attack-strategy-specific.

The attacker payoff uses judge-based attack success when complete end-to-end judge results are available. Otherwise it explicitly uses defense bypass rate as a proxy:

$$U_A(A_i,D_j)=S_{ij}-\lambda C_i^{attack}.$$

Payoff weights are modeling assumptions rather than measured facts. Multiple scenarios are generated for sensitivity analysis in `11_game_theory.ipynb`. Quick data validate this pipeline only.

In [1]:
# =========================
#  1. BLOCK – DEPENDENCIES
# =========================

from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
# =========================
#  2. BLOCK – CONFIGURATION
# =========================

# Use 'auto', 'quick', 'pilot', or 'full'.
RUN_MODE = "full"
RUN_PRIORITY = ["full", "pilot", "quick"]

# Use 'auto', 'judge_asr', or 'pass_rate'.
# Auto uses judge ASR only when every strategy × defense cell contains it.
SUCCESS_SIGNAL_MODE = "auto"

BASE_DEFENDER_SCENARIO = "balanced"
BASE_ATTACK_COST_MODE = "free_prompt_creation"
ATTACKER_COST_WEIGHT = 0.10

DEFENDER_SCENARIOS = {
    "balanced": {
        "security": 0.60,
        "false_positive": 0.20,
        "tokens": 0.10,
        "latency": 0.10,
    },
    "security_heavy": {
        "security": 0.80,
        "false_positive": 0.10,
        "tokens": 0.05,
        "latency": 0.05,
    },
    "false_positive_heavy": {
        "security": 0.45,
        "false_positive": 0.40,
        "tokens": 0.075,
        "latency": 0.075,
    },
    "cost_heavy": {
        "security": 0.40,
        "false_positive": 0.15,
        "tokens": 0.25,
        "latency": 0.20,
    },
}

ATTACK_COST_MODES = [
    "free_prompt_creation",
    "pay_per_execution",
]

for scenario_name, weights in DEFENDER_SCENARIOS.items():
    if not np.isclose(sum(weights.values()), 1.0):
        raise ValueError(f"Defender weights must sum to 1 in scenario: {scenario_name}")

if BASE_DEFENDER_SCENARIO not in DEFENDER_SCENARIOS:
    raise ValueError("Unknown BASE_DEFENDER_SCENARIO.")
if BASE_ATTACK_COST_MODE not in ATTACK_COST_MODES:
    raise ValueError("Unknown BASE_ATTACK_COST_MODE.")


In [3]:
# =========================
#  3. BLOCK – SELECT AND LOAD METRIC FILES
# =========================

PROJECT_ROOT = Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_SUFFIXES = [
    "defense_summary",
    "attack_strategy",
    "attack_pass_matrix",
    "attack_cost_matrix",
]

def metric_path_candidates(mode: str, suffix: str) -> list[Path]:
    filename = f"metrics_{mode}_{suffix}.csv"
    return [RESULTS_DIR / filename, PROJECT_ROOT / filename]

def resolve_mode(mode: str) -> tuple[str, dict[str, Path]]:
    modes = RUN_PRIORITY if mode == "auto" else [mode]
    if mode != "auto" and mode not in RUN_PRIORITY:
        raise ValueError("RUN_MODE must be 'auto', 'quick', 'pilot', or 'full'.")

    for candidate_mode in modes:
        resolved = {}
        for suffix in REQUIRED_SUFFIXES:
            path = next(
                (item for item in metric_path_candidates(candidate_mode, suffix) if item.exists()),
                None,
            )
            if path is None:
                resolved = {}
                break
            resolved[suffix] = path
        if resolved:
            return candidate_mode, resolved

    raise FileNotFoundError(
        "No complete quick, pilot, or full metric-file set was found."
    )

selected_mode, metric_paths = resolve_mode(RUN_MODE)

df_defense = pd.read_csv(metric_paths["defense_summary"])
df_attack = pd.read_csv(metric_paths["attack_strategy"])
df_pass = pd.read_csv(metric_paths["attack_pass_matrix"])
df_defense_cost = pd.read_csv(metric_paths["attack_cost_matrix"])

print("Selected run mode:", selected_mode)
for suffix, path in metric_paths.items():
    print(f"{suffix}: {path}")


Selected run mode: full
defense_summary: C:\work\LLM filtering\TDK\code\results\metrics_full_defense_summary.csv
attack_strategy: C:\work\LLM filtering\TDK\code\results\metrics_full_attack_strategy.csv
attack_pass_matrix: C:\work\LLM filtering\TDK\code\results\metrics_full_attack_pass_matrix.csv
attack_cost_matrix: C:\work\LLM filtering\TDK\code\results\metrics_full_attack_cost_matrix.csv


In [4]:
# =========================
#  4. BLOCK – ALIGN AND VALIDATE MATRICES
# =========================

INDEX_COLUMNS = ["strategy_id", "strategy_name"]

for column in INDEX_COLUMNS:
    if column not in df_pass.columns or column not in df_defense_cost.columns:
        raise ValueError(f"Missing matrix index column: {column}")

pass_matrix = df_pass.set_index(INDEX_COLUMNS).apply(pd.to_numeric, errors="coerce")
defense_cost_matrix = (
    df_defense_cost.set_index(INDEX_COLUMNS).apply(pd.to_numeric, errors="coerce")
)

common_defenses = [
    defense for defense in pass_matrix.columns
    if defense in defense_cost_matrix.columns
]
if not common_defenses:
    raise ValueError("No common defenses were found in the pass and cost matrices.")

pass_matrix = pass_matrix[common_defenses].sort_index()
defense_cost_matrix = defense_cost_matrix.reindex(pass_matrix.index)[common_defenses]

if pass_matrix.isna().any().any():
    raise ValueError("Attack-pass matrix contains missing strategy × defense cells.")
if defense_cost_matrix.isna().any().any():
    raise ValueError("Defense-cost matrix contains missing strategy × defense cells.")
if ((pass_matrix < 0) | (pass_matrix > 1)).any().any():
    raise ValueError("Attack-pass rates must be between 0 and 1.")

required_defense_columns = {
    "defense_id", "f1", "fpr", "mean_operational_latency_seconds"
}
missing_defense_columns = required_defense_columns - set(df_defense.columns)
if missing_defense_columns:
    raise ValueError(f"Missing defense-summary columns: {sorted(missing_defense_columns)}")

defense_lookup = df_defense.set_index("defense_id").reindex(common_defenses)
if defense_lookup.index.has_duplicates or defense_lookup.isna().all(axis=1).any():
    raise ValueError("Defense summary does not uniquely cover every matrix defense.")

print("Attack strategies:", len(pass_matrix.index))
print("Defenses:", len(common_defenses))
display(pass_matrix)


Attack strategies: 3
Defenses: 5


,,D0,D1,D2,D3,D4
strategy_id,strategy_name,,,,,
S1,Fake Directive Manipulation,1.0,0.974684,0.887500,0.325000,0.839506
S2,Stealthy Dependency Injection,1.0,1.000000,0.571429,0.500000,0.714286
S3,Context Hijacking,1.0,1.000000,0.833333,0.333333,0.666667


In [5]:
# =========================
#  5. BLOCK – SUCCESS SIGNAL AND NORMALIZED COMPONENTS
# =========================

def minmax_normalize(values: pd.DataFrame | pd.Series):
    minimum = values.min().min() if isinstance(values, pd.DataFrame) else values.min()
    maximum = values.max().max() if isinstance(values, pd.DataFrame) else values.max()
    if pd.isna(minimum) or pd.isna(maximum):
        raise ValueError("Cannot normalize missing-only values.")
    if np.isclose(maximum, minimum):
        return values * 0.0
    return (values - minimum) / (maximum - minimum)

strategy_asr = df_attack.pivot(
    index=INDEX_COLUMNS,
    columns="defense_id",
    values="attack_success_rate",
).reindex(index=pass_matrix.index, columns=common_defenses)

judge_asr_complete = not strategy_asr.isna().any().any()
if SUCCESS_SIGNAL_MODE == "auto":
    selected_success_signal = "judge_asr" if judge_asr_complete else "pass_rate"
elif SUCCESS_SIGNAL_MODE in {"judge_asr", "pass_rate"}:
    selected_success_signal = SUCCESS_SIGNAL_MODE
else:
    raise ValueError("SUCCESS_SIGNAL_MODE must be 'auto', 'judge_asr', or 'pass_rate'.")

if selected_success_signal == "judge_asr" and not judge_asr_complete:
    raise ValueError("Judge ASR was requested but is incomplete.")

success_matrix = strategy_asr if selected_success_signal == "judge_asr" else pass_matrix
block_matrix = 1.0 - pass_matrix
normalized_defense_tokens = minmax_normalize(defense_cost_matrix)

fpr = pd.to_numeric(defense_lookup["fpr"], errors="coerce")
latency = pd.to_numeric(
    defense_lookup["mean_operational_latency_seconds"], errors="coerce"
)
if fpr.isna().any() or latency.isna().any():
    raise ValueError("FPR or latency is missing for at least one defense.")
normalized_latency = minmax_normalize(latency)

fpr_matrix = pd.DataFrame(
    np.tile(fpr.to_numpy(), (len(pass_matrix.index), 1)),
    index=pass_matrix.index, columns=common_defenses,
)
latency_matrix = pd.DataFrame(
    np.tile(normalized_latency.to_numpy(), (len(pass_matrix.index), 1)),
    index=pass_matrix.index, columns=common_defenses,
)

attack_execution_tokens = (
    df_attack.groupby(INDEX_COLUMNS)["mean_target_prompt_tokens"].mean()
    .reindex(pass_matrix.index)
)
if attack_execution_tokens.isna().any():
    raise ValueError("Attack execution-token proxy is missing for at least one strategy.")
normalized_attack_cost = minmax_normalize(attack_execution_tokens)

print("Selected attacker success signal:", selected_success_signal)
if selected_success_signal == "pass_rate":
    print("WARNING: payoff uses defense bypass as a proxy, not measured end-to-end ASR.")


Selected attacker success signal: judge_asr


In [6]:
# =========================
#  6. BLOCK – PAYOFF FUNCTIONS
# =========================

def build_defender_payoff(weights: dict[str, float]) -> pd.DataFrame:
    return (
        weights["security"] * block_matrix
        - weights["false_positive"] * fpr_matrix
        - weights["tokens"] * normalized_defense_tokens
        - weights["latency"] * latency_matrix
    )

def build_attacker_payoff(cost_mode: str) -> pd.DataFrame:
    if cost_mode == "free_prompt_creation":
        attack_cost = pd.Series(0.0, index=pass_matrix.index)
    elif cost_mode == "pay_per_execution":
        attack_cost = normalized_attack_cost
    else:
        raise ValueError(f"Unknown attack cost mode: {cost_mode}")

    attack_cost_matrix = pd.DataFrame(
        np.tile(attack_cost.to_numpy()[:, None], (1, len(common_defenses))),
        index=pass_matrix.index, columns=common_defenses,
    )
    return success_matrix - ATTACKER_COST_WEIGHT * attack_cost_matrix

base_defender_payoff = build_defender_payoff(
    DEFENDER_SCENARIOS[BASE_DEFENDER_SCENARIO]
)
base_attacker_payoff = build_attacker_payoff(BASE_ATTACK_COST_MODE)

print("Base defender scenario:", BASE_DEFENDER_SCENARIO)
print("Base attacker cost mode:", BASE_ATTACK_COST_MODE)
display(base_defender_payoff.round(4))
display(base_attacker_payoff.round(4))


Base defender scenario: balanced
Base attacker cost mode: free_prompt_creation


,,D0,D1,D2,D3,D4
strategy_id,strategy_name,,,,,
S1,Fake Directive Manipulation,-0.0187,-0.0172,-0.0852,0.2219,-0.0362
S2,Stealthy Dependency Injection,-0.0161,-0.0522,0.1145,0.0980,0.0641
S3,Context Hijacking,0.0000,-0.0313,-0.0387,0.2132,0.0474


,defense_id,D0,D1,D2,D3,D4
strategy_id,strategy_name,,,,,
S1,Fake Directive Manipulation,0.7375,0.7273,0.7324,0.4231,0.7059
S2,Stealthy Dependency Injection,0.8571,0.5714,0.7500,0.3333,0.6000
S3,Context Hijacking,0.6667,0.8000,0.6000,0.5000,1.0000


In [7]:
# =========================
#  7. BLOCK – SENSITIVITY SCENARIOS
# =========================

scenario_rows = []

for defender_scenario, defender_weights in DEFENDER_SCENARIOS.items():
    defender_payoff = build_defender_payoff(defender_weights)

    for attack_cost_mode in ATTACK_COST_MODES:
        attacker_payoff = build_attacker_payoff(attack_cost_mode)

        for strategy_key in pass_matrix.index:
            strategy_id, strategy_name = strategy_key
            for defense_id in common_defenses:
                scenario_rows.append({
                    "run_mode": selected_mode,
                    "success_signal": selected_success_signal,
                    "defender_scenario": defender_scenario,
                    "attack_cost_mode": attack_cost_mode,
                    "strategy_id": strategy_id,
                    "strategy_name": strategy_name,
                    "defense_id": defense_id,
                    "attack_pass_rate": pass_matrix.loc[strategy_key, defense_id],
                    "attack_block_rate": block_matrix.loc[strategy_key, defense_id],
                    "false_positive_rate": fpr_matrix.loc[strategy_key, defense_id],
                    "normalized_defense_tokens": normalized_defense_tokens.loc[strategy_key, defense_id],
                    "normalized_defense_latency": latency_matrix.loc[strategy_key, defense_id],
                    "defender_payoff": defender_payoff.loc[strategy_key, defense_id],
                    "attacker_payoff": attacker_payoff.loc[strategy_key, defense_id],
                    "w_security": defender_weights["security"],
                    "w_false_positive": defender_weights["false_positive"],
                    "w_tokens": defender_weights["tokens"],
                    "w_latency": defender_weights["latency"],
                    "attacker_cost_weight": ATTACKER_COST_WEIGHT,
                })

df_scenarios = pd.DataFrame(scenario_rows)
print("Sensitivity scenarios:", df_scenarios[["defender_scenario", "attack_cost_mode"]].drop_duplicates().shape[0])
display(df_scenarios.head())


Sensitivity scenarios: 8


,run_mode,success_signal,defender_scenario,attack_cost_mode,strategy_id,strategy_name,defense_id,attack_pass_rate,attack_block_rate,false_positive_rate,normalized_defense_tokens,normalized_defense_latency,defender_payoff,attacker_payoff,w_security,w_false_positive,w_tokens,w_latency,attacker_cost_weight
0,full,judge_asr,balanced,free_prompt_creation,S1,Fake Directive Manipulation,D0,1.000000,0.000000,0.000000,0.186714,0.000000,-0.018671,0.737500,0.6,0.2,0.1,0.1,0.1
1,full,judge_asr,balanced,free_prompt_creation,S1,Fake Directive Manipulation,D1,0.974684,0.025316,0.000000,0.114945,0.209244,-0.017229,0.727273,0.6,0.2,0.1,0.1,0.1
2,full,judge_asr,balanced,free_prompt_creation,S1,Fake Directive Manipulation,D2,0.887500,0.112500,0.000000,0.644868,0.882287,-0.085216,0.732394,0.6,0.2,0.1,0.1,0.1
3,full,judge_asr,balanced,free_prompt_creation,S1,Fake Directive Manipulation,D3,0.325000,0.675000,0.010152,0.811044,1.000000,0.221865,0.423077,0.6,0.2,0.1,0.1,0.1
4,full,judge_asr,balanced,free_prompt_creation,S1,Fake Directive Manipulation,D4,0.839506,0.160494,0.000000,0.604865,0.720104,-0.036201,0.705882,0.6,0.2,0.1,0.1,0.1


In [8]:
# =========================
#  8. BLOCK – SANITY CHECKS
# =========================

if len(pass_matrix.index) < 2:
    print("WARNING: Fewer than two attack strategies are available. Game-theoretic attacker choice is trivial.")
if len(common_defenses) < 2:
    print("WARNING: Fewer than two defenses are available. Game-theoretic defender choice is trivial.")
if selected_mode == "quick":
    print("WARNING: Quick-mode payoffs validate the pipeline only and must not support final equilibrium claims.")

print("Aggregate F1 by defense (diagnostic, not repeated in each payoff cell):")
display(defense_lookup[["f1", "fpr"]].round(4))

print("Defender payoff range:", float(base_defender_payoff.min().min()), "to", float(base_defender_payoff.max().max()))
print("Attacker payoff range:", float(base_attacker_payoff.min().min()), "to", float(base_attacker_payoff.max().max()))


Aggregate F1 by defense (diagnostic, not repeated in each payoff cell):


,f1,fpr
defense_id,,
D0,0.0000,0.0000
D1,0.0430,0.0000
D2,0.2453,0.0000
D3,0.7871,0.0102
D4,0.3063,0.0000


Defender payoff range: -0.08521554285472405 to 0.2218651321418235
Attacker payoff range: 0.3333333333333333 to 1.0


In [9]:
# =========================
#  9. BLOCK – SAVE PAYOFF OUTPUTS
# =========================

OUTPUT_PREFIX = RESULTS_DIR / f"payoff_{selected_mode}"
output_files = {
    "defender_payoff": Path(f"{OUTPUT_PREFIX}_defender_matrix.csv"),
    "attacker_payoff": Path(f"{OUTPUT_PREFIX}_attacker_matrix.csv"),
    "scenarios": Path(f"{OUTPUT_PREFIX}_sensitivity_scenarios.csv"),
    "components": Path(f"{OUTPUT_PREFIX}_components.csv"),
}

base_defender_payoff.to_csv(output_files["defender_payoff"])
base_attacker_payoff.to_csv(output_files["attacker_payoff"])
df_scenarios.to_csv(output_files["scenarios"], index=False)

base_components = df_scenarios.loc[
    df_scenarios["defender_scenario"].eq(BASE_DEFENDER_SCENARIO)
].loc[
    lambda frame: frame["attack_cost_mode"].eq(BASE_ATTACK_COST_MODE)
].copy()
base_components.to_csv(output_files["components"], index=False)

for name, path in output_files.items():
    print(f"{name}: {path}")


defender_payoff: C:\work\LLM filtering\TDK\code\results\payoff_full_defender_matrix.csv
attacker_payoff: C:\work\LLM filtering\TDK\code\results\payoff_full_attacker_matrix.csv
scenarios: C:\work\LLM filtering\TDK\code\results\payoff_full_sensitivity_scenarios.csv
components: C:\work\LLM filtering\TDK\code\results\payoff_full_components.csv


## Interpretation and next stage

The base matrices are one explicit utility specification, not the unique correct payoff definition. `11_game_theory.ipynb` should solve the game separately for every saved sensitivity scenario and report whether pure or mixed equilibria change when the weights or attacker-cost model change. If small weight changes produce different equilibria, conclusions must be conditional on the chosen cost and risk assumptions.

The `pay_per_execution` attacker-cost model currently uses normalized target prompt tokens as a proxy. These tokens include the fixed target system prompt, so a later experiment should log attack-text tokens and prompt-generation cost separately if attacker cost becomes a central claim. Latent-space methods and black-box mapping of strategy switches remain future work.